# Candidate Reranking

## Objective
Reorder a plausible candidate set using explicit conversational signals such as budget caps, brand exclusions, use-case hints, and reference-item comparisons, while keeping every ranking change inspectable. Retrieval stays broad at this stage; the purpose here is to show how explicit constraints and preferences should reshape the ordering without hiding the policy behind a black-box ranker.

## Inputs
- retrieval artifacts produced by notebook 02
- curated scenarios from `../data/curated/scenario_suite.json`
- the shared helpers under `functions/`

## Outputs
- baseline candidate sets
- filtered survivors and reranked finalists
- explicit signal and contribution columns
- `../data/candidate_reranking/reranked_candidates.parquet`


## Signal Design

The reranker separates **hard constraints** from **soft ranking signals**. Budget caps, rating floors, exclusions, and cheaper-than-reference checks act as filters. Among the survivors, transparent weighted signals decide the final ordering. This keeps the system steerable and makes it possible to inspect why an item moved up or down.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / "functions").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root containing functions/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from functions.io import load_curated_scenarios, load_retrieval_artifacts, materialize_scenarios
from functions.parsing import build_parser_catalog, parse_user_request
from functions.reranking import rerank_candidates

artifacts = load_retrieval_artifacts(load_encoder_model=True)
paths = artifacts["paths"]
candidate_pool = artifacts["candidate_pool"]
parser_catalog = build_parser_catalog(candidate_pool)
scenario_suite = load_curated_scenarios()
demo_scenarios = materialize_scenarios(scenario_suite["demo_scenarios"], candidate_pool)

paths.candidate_reranking_dir.mkdir(parents=True, exist_ok=True)

print(f"candidate_pool rows  : {len(candidate_pool):,}")
print(f"faiss index vectors  : {artifacts['faiss_index'].ntotal:,}")
print(f"source distribution  : {candidate_pool['source'].value_counts().to_dict()}")
print(f"price coverage       : {candidate_pool['price'].notna().mean():.1%}")
print(f"rating coverage      : {candidate_pool['average_rating'].notna().mean():.1%}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


candidate_pool rows  : 300,000
faiss index vectors  : 300,000
source distribution  : {'electronics': 100000, 'home_and_kitchen': 100000, 'sports_and_outdoors': 100000}
price coverage       : 100.0%
rating coverage      : 100.0%


In [2]:
SIGNAL_SUMMARY = pd.DataFrame(
    [
        ("s_retrieval", "normalized retrieval score", "always on"),
        ("s_rating", "normalized average rating", "always on"),
        ("s_popularity", "log-normalized review count", "always on"),
        ("s_price_fit", "fit to explicit budget or cheaper-than goal", "only if price preference exists"),
        ("s_reference_similarity", "similarity to anchor item", "only if a resolved reference item exists"),
        (
            "s_need_match",
            "lexical support for must-have terms and use-case tokens",
            "only if usable request terms exist",
        ),
    ],
    columns=["Signal", "Meaning", "When it matters"],
)

display(SIGNAL_SUMMARY)

,Signal,Meaning,When it matters
0,s_retrieval,normalized retrieval score,always on
1,s_rating,normalized average rating,always on
2,s_popularity,log-normalized review count,always on
3,s_price_fit,fit to explicit budget or cheaper-than goal,only if price preference exists
4,s_reference_similarity,similarity to anchor item,only if a resolved reference item exists
5,s_need_match,lexical support for must-have terms and use-case tokens,only if usable request terms exist


## Worked Examples

The scenarios below are meant as sanity checks, not proof of global optimality. The important question is whether the ranking shifts feel *defensible* and whether the notebook exposes enough structure to explain those shifts without hiding behind a black-box ranker.


In [3]:
example_labels = [
    "budget_headphones_avoid_beats",
    "reference_cheaper_headphones",
    "induction_frying_pan",
    "lightweight_programming_laptop",
]
selected_scenarios = [scenario for scenario in demo_scenarios if scenario["label"] in example_labels]

worked_examples = []
for scenario in selected_scenarios:
    parsed = parse_user_request(
        scenario["query"],
        parser_catalog,
        reference_item_context=scenario["reference_item_context"],
    )
    result = rerank_candidates(parsed, artifacts, top_k_retrieval=20, top_k_final=5)
    worked_examples.append(
        {
            "label": scenario["label"],
            "title": scenario["title"],
            "query": scenario["query"],
            "parsed_request": parsed,
            "result": result,
        }
    )

weights_summary = pd.DataFrame(
    [{"label": example["label"], **example["result"]["weights"]} for example in worked_examples]
)

display(weights_summary)

,label,retrieval,rating,popularity,price_fit,reference_similarity,need_match
0,budget_headphones_avoid_beats,0.35,0.25,0.05,0.15,0.0,0.2
1,reference_cheaper_headphones,0.35,0.20,0.10,0.15,0.1,0.1
2,induction_frying_pan,0.50,0.20,0.10,0.00,0.0,0.2
3,lightweight_programming_laptop,0.50,0.20,0.10,0.00,0.0,0.2


In [4]:
for example in worked_examples:
    baseline = example["result"]["baseline_candidates"]
    filtered = example["result"]["filtered_candidates"]
    reranked = example["result"]["reranked_candidates"]

    print("=" * 100)
    print(example["label"])
    print(example["query"])
    print("Parsed request")
    print(json.dumps(example["parsed_request"], indent=2, ensure_ascii=False))
    print("Weights")
    print(example["result"]["weights"])

    print("Baseline candidates")
    display(baseline[["parent_asin", "title", "price", "average_rating", "retrieval_score"]].head(5))

    print("Filtered survivors")
    display(
        filtered[
            [
                "parent_asin",
                "title",
                "price",
                "average_rating",
                "retrieval_score",
                "s_retrieval",
                "s_rating",
                "s_popularity",
                "s_price_fit",
                "s_reference_similarity",
                "s_need_match",
            ]
        ].head(5)
    )

    print("Reranked finalists")
    display(
        reranked[
            [
                "reranked_rank",
                "baseline_rank",
                "rank_shift",
                "parent_asin",
                "title",
                "price",
                "average_rating",
                "retrieval_score",
                "final_score",
                "c_retrieval",
                "c_rating",
                "c_popularity",
                "c_price_fit",
                "c_reference_similarity",
                "c_need_match",
            ]
        ]
    )

budget_headphones_avoid_beats
Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats.
Parsed request
{
  "original_query": "Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats.",
  "user_intent": "constrained_search",
  "domain_hint": "electronics",
  "hard_constraints": {
    "max_price": 80.0,
    "min_rating": null,
    "must_include_terms": [
      "headphones"
    ],
    "use_case": null,
    "cheaper_than_reference": false,
    "same_source_as_reference": false
  },
  "soft_preferences": [
    "highly_rated"
  ],
  "reference_items": [],
  "excluded_brands": [
    "Beats"
  ],
  "excluded_terms": [],
  "grounding_needs": [
    "price",
    "rating",
    "brand_constraint"
  ],
  "clarification_needed": false,
  "clarification_questions": []
}
Weights
{'retrieval': 0.35, 'rating': 0.25, 'popularity': 0.05, 'price_fit': 0.15, 'reference_similarity': 0.0, 'need_match': 0.2}
Baseline candidates


,parent_asin,title,price,average_rating,retrieval_score
0,B081RBT9HM,"Black Panther Headphones, Adjustable Headband, Stereo Sound, 3.5Mm Jack, Wired Headphones, Tangle-Free, Volume Control, Foldable, Kids Headphones Over Ear for School Home Travel",19.99,4.3,0.122747
1,B00C4FBQT8,Sony 900MHz Wireless Stereo Noise Reduction Headphones,119.95,3.9,0.106422
2,B0BZ6QTH4S,"PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple",19.99,4.4,0.106393
3,B07DHZMQZS,Jabra Elite 65t Alexa Enabled True Wireless Earbuds with Charging Case IP55 rated - Titanium Black (Renewed),58.97,4.0,0.106393
4,B07NT6BMTK,"Buyer's Point Ultra High Speed HDMI 2.1 Cable CL3 Rated Dynamic HDR 1.8M(6ft) 8K 120Hz, 48Gbps, eARC (1 Pack, Black CL3 Rated)",11.99,4.6,0.106340


Filtered survivors


,parent_asin,title,price,average_rating,retrieval_score,s_retrieval,s_rating,s_popularity,s_price_fit,s_reference_similarity,s_need_match
0,B081RBT9HM,"Black Panther Headphones, Adjustable Headband, Stereo Sound, 3.5Mm Jack, Wired Headphones, Tangle-Free, Volume Control, Foldable, Kids Headphones Over Ear for School Home Travel",19.99,4.3,0.122747,1.000000,0.825,0.003172,0.750125,0.0,1.0
1,B0BZ6QTH4S,"PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple",19.99,4.4,0.106393,0.070581,0.850,0.925688,0.750125,0.0,1.0
2,B0C5RJYKN7,"V8 Bluetooth Headphones, 80 Hours Playtime Wireless Headphones with Deep Bass,Lightweight Foldable Headphones Built-in Mic,HiFi Stereo Sound for Travel Work Laptop PC Cellphone",18.89,4.5,0.105206,0.003096,0.875,0.764038,0.763875,0.0,1.0
3,B09YTGFYZK,"SAMSUNG Galaxy Earbuds Charging Case Cover Compatible with Buds2 Pro, Buds Pro, Buds Live, Buds2, IP67 Rated Water and Dust Resistant Protection, Clear",9.95,4.4,0.105385,0.013248,0.850,0.126018,0.875625,0.0,1.0
4,B0BF2M87TV,"SAMSUNG Galaxy Earbuds Charging Case Cover Compatible with Buds2 Pro, Buds Pro, Buds Live, Buds2, IP67 Rated Water and Dust Resistant Protection, Clear",11.30,4.4,0.105152,0.000000,0.850,0.158946,0.858750,0.0,1.0


Reranked finalists


...

,parent_asin,title,price,average_rating,retrieval_score
0,B01N5E5T48,"ASUS L402SA Portable Lightweight Laptop PC, Intel Dual Core Processor, 4GB RAM, 32GB Flash Storage with Windows 10 with 1 Year Microsoft Office 365 Subscription",188.88,3.4,0.199927
1,B07YHJ9P5N,"Asus Vivobook E203MA Thin and Lightweight 11.6” HD Laptop, Intel Celeron N4000 Processor, 4GB RAM, 64GB eMMC Storage, 802.11AC Wi-Fi, HDMI, USB-C, Win 10",185.12,4.0,0.199251
2,B09NBG2K3G,"Broage 15.6"" FHD Lightweight Laptop Computer, Intel Celeron N4020 up to 2.8GHz, 4GB RAM, 64GB eMMC, WiFi, Bluetooth, USB 3.0, HDMI, Webcam, Microphone, Silver, Windows 10 Home, Online Class Ready",129.00,4.0,0.198655
3,B0001EMA80,Tabletote Black Portable Compact Lightweight Adjustable Height Laptop Notebook Computer Stand Table,49.99,4.3,0.196932
4,B00LMIPTFK,"Spektrum TX/RX USB Programming Cable, Laptop",24.99,4.5,0.196393


Filtered survivors


,parent_asin,title,price,average_rating,retrieval_score,s_retrieval,s_rating,s_popularity,s_price_fit,s_reference_similarity,s_need_match
0,B07YHJ9P5N,"Asus Vivobook E203MA Thin and Lightweight 11.6” HD Laptop, Intel Celeron N4000 Processor, 4GB RAM, 64GB eMMC Storage, 802.11AC Wi-Fi, HDMI, USB-C, Win 10",185.12,4.0,0.199251,0.924411,0.750,0.430747,0.0,0.0,0.666667
1,B01N5E5T48,"ASUS L402SA Portable Lightweight Laptop PC, Intel Dual Core Processor, 4GB RAM, 32GB Flash Storage with Windows 10 with 1 Year Microsoft Office 365 Subscription",188.88,3.4,0.199927,1.000000,0.600,0.118783,0.0,0.0,0.666667
2,B09NBG2K3G,"Broage 15.6"" FHD Lightweight Laptop Computer, Intel Celeron N4020 up to 2.8GHz, 4GB RAM, 64GB eMMC, WiFi, Bluetooth, USB 3.0, HDMI, Webcam, Microphone, Silver, Windows 10 Home, Online Class Ready",129.00,4.0,0.198655,0.857719,0.750,0.088837,0.0,0.0,0.666667
3,B0001EMA80,Tabletote Black Portable Compact Lightweight Adjustable Height Laptop Notebook Computer Stand Table,49.99,4.3,0.196932,0.664883,0.825,0.491945,0.0,0.0,0.666667
4,B00LMIPTFK,"Spektrum TX/RX USB Programming Cable, Laptop",24.99,4.5,0.196393,0.604658,0.875,0.394784,0.0,0.0,0.666667


Reranked finalists


,reranked_rank,baseline_rank,rank_shift,parent_asin,title,price,average_rating,retrieval_score,final_score,c_retrieval,c_rating,c_popularity,c_price_fit,c_reference_similarity,c_need_match
0,1,2,1,B07YHJ9P5N,"Asus Vivobook E203MA Thin and Lightweight 11.6” HD Laptop, Intel Celeron N4000 Processor, 4GB RAM, 64GB eMMC Storage, 802.11AC Wi-Fi, HDMI, USB-C, Win 10",185.12,4.0,0.199251,0.788613,0.462205,0.150,0.043075,0.0,0.0,0.133333
1,2,1,-1,B01N5E5T48,"ASUS L402SA Portable Lightweight Laptop PC, Intel Dual Core Processor, 4GB RAM, 32GB Flash Storage with Windows 10 with 1 Year Microsoft Office 365 Subscription",188.88,3.4,0.199927,0.765212,0.500000,0.120,0.011878,0.0,0.0,0.133333
2,3,3,0,B09NBG2K3G,"Broage 15.6"" FHD Lightweight Laptop Computer, Intel Celeron N4020 up to 2.8GHz, 4GB RAM, 64GB eMMC, WiFi, Bluetooth, USB 3.0, HDMI, Webcam, Microphone, Silver, Windows 10 Home, Online Class Ready",129.00,4.0,0.198655,0.721076,0.428859,0.150,0.008884,0.0,0.0,0.133333
3,4,4,0,B0001EMA80,Tabletote Black Portable Compact Lightweight Adjustable Height Laptop Notebook Computer Stand Table,49.99,4.3,0.196932,0.679969,0.332441,0.165,0.049195,0.0,0.0,0.133333
4,5,5,0,B00LMIPTFK,"Spektrum TX/RX USB Programming Cable, Laptop",24.99,4.5,0.196393,0.650141,0.302329,0.175,0.039478,0.0,0.0,0.133333


In [5]:
export_rows = []
for example in worked_examples:
    reranked = example["result"]["reranked_candidates"].copy()
    reranked["example_label"] = example["label"]
    reranked["query"] = example["query"]
    export_rows.append(reranked)

reranked_export = pd.concat(export_rows, ignore_index=True)
output_path = paths.candidate_reranking_dir / "reranked_candidates.parquet"
reranked_export.to_parquet(output_path, index=False)
print(f"saved {len(reranked_export)} rows to {output_path}")
display(reranked_export.head())

saved 20 rows to /home/imont/ConversationalRecommenderSystem/data/candidate_reranking/reranked_candidates.parquet


,retrieval_row_id,parent_asin,source,main_category,title,store,average_rating,rating_number,price,retrieval_text,retrieval_text_chars,lexical_overlap,retrieval_score,baseline_rank,passes_constraints,s_retrieval,s_rating,s_popularity,s_price_fit,s_reference_similarity,s_need_match,w_retrieval,c_retrieval,w_rating,c_rating,w_popularity,c_popularity,w_price_fit,c_price_fit,w_reference_similarity,c_reference_similarity,w_need_match,c_need_match,final_score,reranked_rank,rank_shift,example_label,query
0,75839,B081RBT9HM,electronics,All Electronics,"Black Panther Headphones, Adjustable Headband, Stereo Sound, 3.5Mm Jack, Wired Headphones, Tangle-Free, Volume Control, Foldable, Kids Headphones Over Ear for School Home Travel",Kids Disney Headphones,4.3,243,19.99,"Black Panther Headphones, Adjustable Headband, Stereo Sound, 3.5Mm Jack, Wired Headphones, Tangle-Free, Volume Control, Foldable, Kids Headphones Over Ear for School Home Travel Black Panther Head...",487,4.0,0.122747,1,True,1.000000,0.825,0.003172,0.750125,0.0,1.0,0.35,0.350000,0.25,0.20625,0.05,0.000159,0.15,0.112519,0.0,0.0,0.2,0.2,0.868927,1,0,budget_headphones_avoid_beats,"Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats."
1,2739,B0BZ6QTH4S,electronics,All Electronics,"PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple",Philips,4.4,8915,19.99,"PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple ...",310,3.0,0.106393,3,True,0.070581,0.850,0.925688,0.750125,0.0,1.0,0.35,0.024703,0.25,0.21250,0.05,0.046284,0.15,0.112519,0.0,0.0,0.2,0.2,0.596007,2,1,budget_headphones_avoid_beats,"Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats."
2,5716,B0C5RJYKN7,electronics,All Electronics,"V8 Bluetooth Headphones, 80 Hours Playtime Wireless Headphones with Deep Bass,Lightweight Foldable Headphones Built-in Mic,HiFi Stereo Sound for Travel Work Laptop PC Cellphone",fojep,4.5,4745,18.89,"V8 Bluetooth Headphones, 80 Hours Playtime Wireless Headphones with Deep Bass,Lightweight Foldable Headphones Built-in Mic,HiFi Stereo Sound for Travel Work Laptop PC Cellphone V8 Bluetooth Headph...",468,3.0,0.105206,14,True,0.003096,0.875,0.764038,0.763875,0.0,1.0,0.35,0.001084,0.25,0.21875,0.05,0.038202,0.15,0.114581,0.0,0.0,0.2,0.2,0.572617,3,11,budget_headphones_avoid_beats,"Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats."
3,54484,B09YTGFYZK,electronics,Cell Phones & Accessories,"SAMSUNG Galaxy Earbuds Charging Case Cover Compatible with Buds2 Pro, Buds Pro, Buds Live, Buds2, IP67 Rated Water and Dust Resistant Protection, Clear",SAMSUNG,4.4,393,9.95,"SAMSUNG Galaxy Earbuds Charging Case Cover Compatible with Buds2 Pro, Buds Pro, Buds Live, Buds2, IP67 Rated Water and Dust Resistant Protection, Clear SAMSUNG Galaxy Earbuds Charging Case Cover C...",393,3.0,0.105385,12,True,0.013248,0.850,0.126018,0.875625,0.0,1.0,0.35,0.004637,0.25,0.21250,0.05,0.006301,0.15,0.131344,0.0,0.0,0.2,0.2,0.554781,4,8,budget_headphones_avoid_beats,"Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats."
4,49568,B0BF2M87TV,electronics,Cell Phones & Accessories,"SAMSUNG Galaxy Earbuds Charging Case Cover Compatible with Buds2 Pro, Buds Pro, Buds Live, Buds2, IP67 Rated Water and Dust Resistant Protection, Clear",SAMSUNG,4.4,447,11.30,"SAMSUNG Galaxy Earbuds Charging Case Cover Compatible with Buds2 Pro, Buds Pro, Buds Live, Buds2, IP67 Rated Water and Dust Resistant Protection, Clear SAMSUNG Galaxy Earbuds Charging Case Cover C...",393,3.0,0.105152,15,True,0.000000,0.850,0.158946,0.858750,0.0,1.0,0.35,0.000000,0.25,0.21250,0.05,0.007947,0.15,0.128812,0.0,0.0,0.2,0.2,0.549260,5,10,budget_headphones_avoid_beats,"Recommend wireless bluetooth headphones under 8

## Limits Of The Current Reranker

This reranker is still deliberately simple. Hard-coded patterns remain brittle, metadata remains incomplete, and lexical need-match cannot capture subtle preference trade-offs. At this stage, the goal is transparent policy correction rather than a less interpretable ranking model.
